# 巨大な辞書の語をテキストから一括で探す (Aho-Corasick)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/02_string/aho_corasick.ipynb)

NLP クックブック 02-26 ／ 文字列処理と正規化

数万〜数百万語の辞書（商品名、病名、NG ワードなど）に載っている語を、長いテキストから高速にすべて見つけたい。

In [1]:
!pip install -q pyahocorasick

In [2]:
import ahocorasick

terms = {"東京": "LOC", "東京都": "LOC", "京都": "LOC", "東京都立大学": "ORG", "自然言語処理": "TERM", "言語": "TERM", "機械翻訳": "TERM"}
A = ahocorasick.Automaton()
for term, label in terms.items():
    A.add_word(term, (term, label))
A.make_automaton()

text = "東京都立大学では自然言語処理と機械翻訳を研究している。京都にも行った。"
for end, (term, label) in A.iter(text):          # 重なりも含めてすべての一致
    print(end - len(term) + 1, end + 1, term, label)

0 2 東京 LOC
0 3 東京都 LOC
1 3 京都 LOC
0 6 東京都立大学 ORG
10 12 言語 TERM
8 14 自然言語処理 TERM
15 19 機械翻訳 TERM
27 29 京都 LOC


In [3]:
# 最長一致だけ残す（重なったら長い語を優先）
for end, (term, label) in A.iter_long(text):
    print(end - len(term) + 1, end + 1, term, label)

0 6 東京都立大学 ORG
8 14 自然言語処理 TERM
15 19 機械翻訳 TERM
27 29 京都 LOC


In [4]:
# 語ごとにテキスト全体を調べる単純な in 判定と速度を比べる
# （in 判定は 5 万語だと時間がかかるので、500 語で測って 5 万語ぶんに換算する）
import random, time

random.seed(0)
vocab = ["".join(random.choices("あいうえおかきくけこさしすせそ", k=random.randint(3, 6))) for _ in range(50000)]
big = "".join(random.choices("あいうえおかきくけこさしすせそ", k=200000))
B = ahocorasick.Automaton()
for w in vocab:
    B.add_word(w, w)
B.make_automaton()

t = time.perf_counter(); n1 = sum(1 for _ in B.iter(big)); t1 = time.perf_counter() - t
t = time.perf_counter(); n2 = sum(1 for w in vocab[:500] if w in big); t2 = time.perf_counter() - t
print(f"Aho-Corasick (5 万語): {t1:.3f} 秒")
print(f"単純な in 判定 (500 語): {t2:.3f} 秒 → 5 万語なら約 {t2 * 100:.1f} 秒")

Aho-Corasick (5 万語): 0.051 秒
単純な in 判定 (500 語): 0.042 秒 → 5 万語なら約 4.2 秒


- テキストを 1 回走査するだけで、辞書の語数に関係なくすべての一致を列挙できる。
- 「東京都」の中の「京都」のような誤一致は最長一致でも完全には防げない。形態素境界と照合すると精度が上がる。